# Silver Layer — Metadata Transformation

**Catalog:** metadata_governance

**Schema:** silver

**Table:** silver_metadata

**Layer:** Silver (cleaned, standardized, and validated data with quality rules applied)

**Source:** Bronze Layer Table — metadata_governance.bronze.bronze_metadata

**Purpose:** The Silver layer processes raw Bronze data by applying data cleaning, standardization, and validation rules to produce a trusted dataset ready for analytics and reporting.

## Step 1 — Read Metadata from Bronze Layer

Read the metadata dataset from the Bronze layer and verify that all records are successfully loaded before transformation.

In [0]:
bronze_df = spark.table("metadata_governance.bronze.bronze_metadata")

bronze_count = bronze_df.count()
print(f"Bronze rows loaded: {bronze_count}")

## Step 2 — Clean and Standardize Metadata

Standardize text fields by removing leading and trailing spaces and converting values to lowercase to ensure consistent formatting across the dataset.

In [0]:
from pyspark.sql.functions import col, trim, lower

text_cols = [
    "column_name","column_desc", "term_name", "term_description",
    "security_classification",
    "term_subdomain", "data_steward","table_name", "table_desc",
    "table_owner_in_source", "schema_name", "database_name",
    "system_name", "location", "tag_name", "tag_value", "certification_level"
]

df_cleaned = bronze_df

for c in text_cols:
    if c in df_cleaned.columns:
        df_cleaned = df_cleaned.withColumn(
            c,
            lower(trim(col(c)))
        )

print(f"Cleaned rows: {df_cleaned.count()}")

## Step 3 — Apply Data Quality Validation Rules

Validate required metadata fields and separate invalid records into a quarantine dataset while retaining valid records for further processing.

In [0]:
from pyspark.sql.functions import col, isnull, when, concat_ws, lit

# INVALID ROWS
df_invalid = df_cleaned.filter(
    isnull(col("table_name")) | (col("table_name") == "") |
    isnull(col("schema_name")) | (col("schema_name") == "") |
    isnull(col("database_name")) | (col("database_name") == "")
).withColumn(
    "failure_reason",
    concat_ws(", ",
        when(isnull(col("table_name")) | (col("table_name") == ""),lit("table_name is null")),
        when(isnull(col("schema_name")) | (col("schema_name") == ""),lit("schema_name is null")),
        when(isnull(col("database_name")) | (col("database_name") == ""), lit("database_name is null"))
    )
)

# VALID ROWS
df_valid = df_cleaned.filter(
    col("table_name").isNotNull() & (col("table_name") != "") &
    col("schema_name").isNotNull() & (col("schema_name") != "") &
    col("database_name").isNotNull() & (col("database_name") != "")
)

# COUNTS
valid_count = df_valid.count()
invalid_count = df_invalid.count()

print(f"Valid rows: {valid_count}")
print(f"Invalid rows: {invalid_count}")
print(f"Total check: {valid_count + invalid_count}")
assert valid_count + invalid_count == bronze_count, "Row count mismatch"

In [0]:
display(df_invalid.groupBy("failure_reason").count())

## Step 4 — Save Validated Metadata to Silver Layer

Store the validated metadata dataset in the Silver layer as a trusted source for analytics, reporting, and governance activities.



In [0]:
df_valid.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("metadata_governance.silver.silver_metadata")

print(f"Silver table written: {df_valid.count()} rows")

## Step 5 — Save Quarantine Records and Generate Compliance Alert

Store invalid records in a quarantine table, generate a failure summary, and create a compliance alert when validation rules are violated.

In [0]:

if invalid_count > 0:
    
    df_invalid.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("metadata_governance.silver.bronze_metadata_quarantine")

    print(f"Alert: {invalid_count} rows moved to quarantine!")

    print("Failure breakdown:")
    display(
        df_invalid.groupBy("failure_reason")
        .count()
        .orderBy("count", ascending=False)
    )

    alert_msg = f"[COMPLIANCE ALERT] {invalid_count} rows failed validation. Table: silver_metadata_quarantine. Date: {spark.sql('SELECT current_date()').collect()[0][0]}"

    print(alert_msg)

else:
    print("All rows passed validation")

print("Notebook execution completed successfully.")

## Step 6 — Validate Silver Output

> Verify that the Silver table was successfully created by checking the row count, schema, and sample records.


In [0]:
silver_df = spark.table("metadata_governance.silver.silver_metadata")
print(f"Silver final row count: {silver_df.count()}")
silver_df.printSchema()
display(silver_df.limit(5))